# t-SNE: Análisis de Clusters de Conversaciones

Proyecta las consultas de usuarios en 2D usando t-SNE para visualizar patrones de conversación del agente Riopaila Castilla.

In [ ]:
import json
import requests
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from pathlib import Path

BASE = Path.cwd().parent
LOG_PATH = BASE / "data" / "logs" / "conversations.jsonl"

In [ ]:
# Cargar logs
entries = []
if LOG_PATH.exists():
    with open(LOG_PATH, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                entries.append(json.loads(line))

print(f"Cargadas {len(entries)} entradas de conversación")

In [ ]:
# Generar embeddings para cada consulta usando Ollama
def get_embedding(text):
    resp = requests.post(
        "http://localhost:11434/api/embeddings",
        json={"model": "mxbai-embed-large", "prompt": text},
        timeout=30
    )
    resp.raise_for_status()
    return resp.json()["embedding"]

embeddings = []
labels = []
errors = []

for i, e in enumerate(entries):
    print(f"Procesando {i+1}/{len(entries)}: {e['query'][:50]}...")
    try:
        emb = get_embedding(e["query"])
        embeddings.append(emb)
        labels.append(e)
    except Exception as ex:
        errors.append((i, str(ex)))

print(f"\nEmbeddings generados: {len(embeddings)}")
if errors:
    print(f"Errores: {len(errors)}")

In [ ]:
# Reducción con t-SNE
X = np.array(embeddings)
tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(X)-1))
coords = tsne.fit_transform(X)
print(f"Forma original: {X.shape}")
print(f"Proyectado a: {coords.shape}")

In [ ]:
# Colorear por categoría (error = rojo, normal = azul)
colores = ["red" if e["error"] else "steelblue" for e in labels]

plt.figure(figsize=(12, 8))
scatter = plt.scatter(coords[:, 0], coords[:, 1], c=colores, alpha=0.7, s=60)

# Anotar algunas consultas
for i, e in enumerate(labels):
    if i % max(1, len(labels)//10) == 0:
        txt = e["query"][:40] + ("..." if len(e["query"]) > 40 else "")
        plt.annotate(txt, (coords[i, 0], coords[i, 1]), fontsize=8, alpha=0.8)

plt.title("Clusters de Conversaciones - Riopaila Castilla")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")

# Leyenda
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='steelblue', markersize=10, label='Normal'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markersize=10, label='Error'),
]
plt.legend(handles=legend_elements)

plt.tight_layout()
plt.savefig(BASE / "data" / "logs" / "tsne_clusters.png", dpi=150)
plt.show()
print(f"Gráfico guardado en: {BASE / 'data' / 'logs' / 'tsne_clusters.png'}")

In [ ]:
# Análisis: ¿qué revelan los clusters?
from collections import Counter

total = len(labels)
errores = sum(1 for e in labels if e["error"])
con_nombre = sum(1 for e in labels if e["has_name"])

print(f"=== Resumen de Conversaciones ===")
print(f"Total: {total}")
print(f"Exitosas: {total - errores}")
print(f"Con error: {errores} ({errores/total*100:.1f}%)")
if total > 0:
    print(f"Con nombre de usuario: {con_nombre} ({con_nombre/total*100:.1f}%)")

# Palabras más frecuentes en consultas exitosas vs con error
exitosas = [e["query"].lower() for e in labels if not e["error"]]
fallidas = [e["query"].lower() for e in labels if e["error"]]

print(f"\nEjemplos de consultas con error:")
for e in labels:
    if e["error"]:
        print(f"  - {e['query'][:80]}")

## Interpretación

- **Clusters**: Cada grupo representa consultas semánticamente similares.
- **Puntos rojos**: Consultas donde el agente falló. Si están agrupados, indican un tema problemático.
- **Puntos azules dispersos**: Consultas diversas que el agente manejó correctamente.
- Si los errores aparecen en un cluster específico, hay que mejorar el FAQ o los documentos para ese tema.